# Hyperparameter Tuning (Optuna)
Tune only XGBoost on the selected feature set from feature engineering.


In [6]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from xgboost import XGBRegressor
from car_pipeline import OutlierDetector, FeatureEngineeringTransformer

import optuna


In [7]:
# Same selected feature set as in feature-engineering notebook
DROP_COLUMNS = ['state', 'size', 'year', 'location_cluster']

RESULTS_DIR = Path('.cache/engineered')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Load and split data
data = pd.read_parquet('.cache/vehicles_cleaned.parquet').copy()
data = data.drop(columns=['region']).dropna(subset=['price'])

X = data.drop(columns=['price'])
y = data['price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train-only outlier fitting; then apply learned price filter to test labels for consistent target quality
detector = OutlierDetector().fit(X_train, y_train)
X_train, y_train = detector.transform(X_train, y_train)
X_test, y_test = detector.transform(X_test, y_test, filter_price=True, filter_odometer=False)


In [8]:
def build_xgb_pipeline(params=None, drop_columns=DROP_COLUMNS):
    params = params or {}

    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', drop='first'))
    ])

    preprocessor = ColumnTransformer([
        ('num', numeric_transformer, make_column_selector(dtype_include=np.number)),
        ('cat', categorical_transformer, make_column_selector(dtype_exclude=np.number))
    ])

    model = XGBRegressor(
        random_state=42,
        n_jobs=4,
        tree_method='hist',
        objective='reg:squarederror',
        **params
    )

    return Pipeline([
        ('features', FeatureEngineeringTransformer(n_clusters=50, drop_columns=drop_columns)),
        ('preprocessor', preprocessor),
        ('model', model)
    ])


In [9]:
cv = KFold(n_splits=3, shuffle=True, random_state=42)

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 1400),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'max_depth': trial.suggest_int('max_depth', 4, 12),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma': trial.suggest_float('gamma', 0.0, 5.0),
    }

    pipe = build_xgb_pipeline(params=params)
    scores = cross_validate(
        pipe,
        X_train,
        y_train,
        cv=cv,
        scoring={'mae': 'neg_mean_absolute_error', 'r2': 'r2'},
        n_jobs=1
    )

    mae = -scores['test_mae'].mean()
    trial.set_user_attr('cv_r2', float(scores['test_r2'].mean()))
    return mae


In [10]:
study_path = RESULTS_DIR / 'optuna_xgb_selected_study.joblib'

if study_path.exists():
    study = joblib.load(study_path)
else:
    study = optuna.create_study(direction='minimize', study_name='xgb_selected_features')
    study.optimize(objective, n_trials=30, show_progress_bar=True)
    joblib.dump(study, study_path)

print('Best CV MAE:', study.best_value)
print('Best params:', study.best_params)


[I 2026-10-05 20:26:47,423] A new study created in memory with name: xgb_selected_features
  0%|          | 0/30 [00:00<?, ?it/s]c:\Users\Luka\Documents\Projects\car-leasing\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
Best trial: 0. Best value: 2222.84:   3%|▎         | 1/30 [00:39<19:07, 39.58s/it]

[I 2026-10-05 20:27:27,006] Trial 0 finished with value: 2222.835205078125 and parameters: {'n_estimators': 483, 'learning_rate': 0.06820768023234056, 'max_depth': 11, 'colsample_bytree': 0.6871253751536364, 'gamma': 0.8067423970143217}. Best is trial 0 with value: 2222.835205078125.


c:\Users\Luka\Documents\Projects\car-leasing\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
Best trial: 1. Best value: 1832.77:   7%|▋         | 2/30 [01:39<23:58, 51.36s/it]

[I 2026-10-05 20:28:26,612] Trial 1 finished with value: 1832.766845703125 and parameters: {'n_estimators': 910, 'learning_rate': 0.13446378126071196, 'max_depth': 11, 'colsample_bytree': 0.5056698002888023, 'gamma': 1.4472308559322333}. Best is trial 1 with value: 1832.766845703125.


c:\Users\Luka\Documents\Projects\car-leasing\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
Best trial: 1. Best value: 1832.77:  10%|█         | 3/30 [02:32<23:27, 52.12s/it]

...

[I 2026-10-05 20:55:19,277] Trial 27 finished with value: 1789.7431640625 and parameters: {'n_estimators': 1244, 'learning_rate': 0.17392398063706369, 'max_depth': 10, 'colsample_bytree': 0.9544702962913276, 'gamma': 4.28242180063097}. Best is trial 20 with value: 1693.1012369791667.


c:\Users\Luka\Documents\Projects\car-leasing\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
Best trial: 20. Best value: 1693.1:  97%|█████████▋| 29/30 [29:53<01:16, 76.76s/it]

[I 2026-10-05 20:56:41,202] Trial 28 finished with value: 1841.490234375 and parameters: {'n_estimators': 1358, 'learning_rate': 0.06501713815151146, 'max_depth': 11, 'colsample_bytree': 0.9910740422384908, 'gamma': 4.424049445304087}. Best is trial 20 with value: 1693.1012369791667.


c:\Users\Luka\Documents\Projects\car-leasing\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
Best trial: 20. Best value: 1693.1: 100%|██████████| 30/30 [31:22<00:00, 62.75s/it]

[I 2026-10-05 20:58:09,863] Trial 29 finished with value: 1736.0696207682292 and parameters: {'n_estimators': 1179, 'learning_rate': 0.07857741414436853, 'max_depth': 12, 'colsample_bytree': 0.7945421242103742, 'gamma': 4.3753307227867415}. Best is trial 20 with value: 1693.1012369791667.
Best CV MAE: 1693.1012369791667
Best params: {'n_estimators': 1341, 'learning_rate': 0.13503616262364118, 'max_depth': 12, 'colsample_bytree': 0.9082405119774818, 'gamma': 3.501540694321212}


In [11]:
best_pipe = build_xgb_pipeline(params=study.best_params).fit(X_train, y_train)
y_pred = best_pipe.predict(X_test)

final_results = {
    'model': 'gradient_boosting_optuna',
    'drop_columns': DROP_COLUMNS,
    'test_mse': mean_squared_error(y_test, y_pred),
    'test_rmse': mean_squared_error(y_test, y_pred) ** 0.5,
    'test_mae': mean_absolute_error(y_test, y_pred),
    'test_r2': r2_score(y_test, y_pred),
    'best_cv_mae': study.best_value,
    'best_cv_r2': study.best_trial.user_attrs.get('cv_r2')
}

joblib.dump(
    {'best_params': study.best_params, 'results': final_results},
    RESULTS_DIR / 'xgb_optuna_selected_final.joblib'
)

final_results


{'model': 'gradient_boosting_optuna',
 'drop_columns': ['state', 'size', 'year', 'location_cluster'],
 'test_mse': 12364291.0,
 'test_rmse': 3516.2893794453266,
 'test_mae': 1501.498046875,
 'test_r2': 0.935161828994751,
 'best_cv_mae': 1693.1012369791667,
 'best_cv_r2': 0.9258095026016235}

In [12]:
# CV vs test consistency check
cv_rows = pd.DataFrame(study.trials_dataframe(attrs=('number', 'value', 'user_attrs')) )
cv_rows = cv_rows[cv_rows['value'].notna()]
cv_mae_mean = float(cv_rows['value'].mean())
cv_mae_std = float(cv_rows['value'].std(ddof=0))
cv_r2_mean = float(cv_rows['user_attrs_cv_r2'].mean())
cv_r2_std = float(cv_rows['user_attrs_cv_r2'].std(ddof=0))

test_mae = float(final_results['test_mae'])
test_r2 = float(final_results['test_r2'])

mae_ok = test_mae <= (cv_mae_mean + cv_mae_std)
r2_ok = test_r2 >= (cv_r2_mean - cv_r2_std)

summary = {
    'cv_mae_mean': cv_mae_mean,
    'cv_mae_std': cv_mae_std,
    'test_mae': test_mae,
    'cv_r2_mean': cv_r2_mean,
    'cv_r2_std': cv_r2_std,
    'test_r2': test_r2,
    'mae_consistent': mae_ok,
    'r2_consistent': r2_ok,
    'overall_consistent': bool(mae_ok and r2_ok),
}

pd.Series(summary)


cv_mae_mean           2208.563097
cv_mae_std             765.976857
test_mae              1501.498047
cv_r2_mean               0.903955
cv_r2_std                0.043091
test_r2                  0.935162
mae_consistent               True
r2_consistent                True
overall_consistent           True
dtype: object

In [13]:
# Professional model breakdown (run after best_pipe/study/final_results cells)
import numpy as np
import pandas as pd
from sklearn.metrics import median_absolute_error

# Ensure predictions are in sync with current best pipeline
y_pred = best_pipe.predict(X_test)

eval_df = X_test.copy()
eval_df['y_true'] = y_test.values
eval_df['y_pred'] = y_pred
eval_df['abs_error'] = (eval_df['y_true'] - eval_df['y_pred']).abs()
eval_df['pct_error'] = eval_df['abs_error'] / eval_df['y_true'].clip(lower=1)

# Core + robust metrics
median_price = float(eval_df['y_true'].median())
report = {
    'n_test_rows': int(len(eval_df)),
    'median_price': median_price,
    'mean_price': float(eval_df['y_true'].mean()),
    'test_mae': float(eval_df['abs_error'].mean()),
    'test_medae': float(median_absolute_error(eval_df['y_true'], eval_df['y_pred'])),
    'test_rmse': float(np.sqrt(np.mean((eval_df['y_true'] - eval_df['y_pred']) ** 2))),
    'test_r2': float(final_results['test_r2']),
    'relative_mae_vs_median_price': float(eval_df['abs_error'].mean() / median_price),
    'mape_percent': float(100 * eval_df['pct_error'].mean()),
    'error_p50': float(eval_df['abs_error'].quantile(0.50)),
    'error_p90': float(eval_df['abs_error'].quantile(0.90)),
    'error_p95': float(eval_df['abs_error'].quantile(0.95)),
    'error_p99': float(eval_df['abs_error'].quantile(0.99)),
}

# CV stability against best Optuna trial
if len(study.trials) > 0 and study.best_trial is not None:
    report['best_cv_mae'] = float(study.best_value)
    report['best_cv_r2'] = float(study.best_trial.user_attrs.get('cv_r2', np.nan))
    report['mae_gap_test_minus_cv'] = report['test_mae'] - report['best_cv_mae']
    report['r2_gap_test_minus_cv'] = report['test_r2'] - report['best_cv_r2']

# Price-band performance (fairness across cheap/mid/expensive vehicles)
price_band = pd.qcut(eval_df['y_true'], q=5, labels=['Q1_low', 'Q2', 'Q3', 'Q4', 'Q5_high'])
band_perf = (
    eval_df.assign(price_band=price_band)
    .groupby('price_band', observed=False)
    .agg(n=('y_true','size'), mae=('abs_error','mean'), medae=('abs_error','median'), mape=('pct_error','mean'))
    .sort_index()
 )
band_perf['mape'] = 100 * band_perf['mape']

# Segment performance on key business slices
def topk_segment_mae(df, col, k=10):
    if col not in df.columns:
        return pd.DataFrame(columns=[col, 'n', 'mae', 'mape'])
    top = df[col].value_counts(dropna=False).head(k).index
    out = (df[df[col].isin(top)]
           .groupby(col, dropna=False)
           .agg(n=('y_true','size'), mae=('abs_error','mean'), mape=('pct_error','mean'))
           .sort_values('mae', ascending=False))
    out['mape'] = 100 * out['mape']
    return out

segment_reports = {
    'manufacturer_top10': topk_segment_mae(eval_df, 'manufacturer', k=10),
    'fuel': topk_segment_mae(eval_df, 'fuel', k=10),
    'condition': topk_segment_mae(eval_df, 'condition', k=10),
    'state_top10': topk_segment_mae(eval_df, 'state', k=10),
}

# Residual bias checks (should be near 0)
eval_df['residual'] = eval_df['y_true'] - eval_df['y_pred']
bias = {
    'mean_residual': float(eval_df['residual'].mean()),
    'median_residual': float(eval_df['residual'].median()),
    'overprediction_rate_percent': float(100 * (eval_df['residual'] < 0).mean()),
}

# Save full report artifacts
model_report = {
    'summary': report,
    'bias': bias,
    'band_performance': band_perf,
    'segment_reports': segment_reports,
}

joblib.dump(model_report, RESULTS_DIR / 'xgb_professional_report.joblib')
band_perf.to_csv(RESULTS_DIR / 'xgb_price_band_report.csv')
for name, df_seg in segment_reports.items():
    df_seg.to_csv(RESULTS_DIR / f'xgb_segment_{name}.csv')

print('=== SUMMARY ===')
display(pd.Series(report).round(6))
print('\n=== BIAS CHECK ===')
display(pd.Series(bias).round(6))
print('\n=== PRICE-BAND PERFORMANCE (quintiles) ===')
display(band_perf.round(3))
print('\n=== WORST MAE SEGMENTS (top categories) ===')
for name, df_seg in segment_reports.items():
    print(f'-- {name} --')
    display(df_seg.head(5).round(3))


=== SUMMARY ===


n_test_rows                     71178.000000
median_price                    15999.000000
mean_price                      19419.453581
test_mae                         1501.497937
test_medae                        376.442383
test_rmse                        3516.289339
test_r2                             0.935162
relative_mae_vs_median_price        0.093849
mape_percent                       22.071810
error_p50                         376.442383
error_p90                        3940.879980
error_p95                        6427.786426
error_p99                       15193.283340
best_cv_mae                      1693.101237
best_cv_r2                          0.925810
mae_gap_test_minus_cv            -191.603300
r2_gap_test_minus_cv                0.009352
dtype: float64


=== BIAS CHECK ===


mean_residual                 -84.275668
median_residual                -3.201172
overprediction_rate_percent    50.934277
dtype: float64


=== PRICE-BAND PERFORMANCE (quintiles) ===


...

,n,mae,mape
fuel,,,
diesel,4647,2640.842,36.934
electric,296,2527.741,44.295
NaN,478,2097.222,11.476
gas,59378,1448.699,21.691
hybrid,980,1266.005,13.789


-- condition --


,n,mae,mape
condition,,,
new,185,4805.264,94.291
salvage,84,3848.465,137.566
like new,3633,2536.213,27.290
fair,850,1897.250,64.581
excellent,17136,1626.117,18.332


-- state_top10 --


,n,mae,mape
state,,,
or,2361,1993.698,83.061
tx,3949,1962.890,30.761
wa,2200,1845.217,29.163
ca,8101,1675.949,22.204
fl,4876,1545.961,19.526


In [14]:
# Traffic-light production readiness check
# Tunable thresholds (edit to your business tolerance)
THRESHOLDS = {
    'relative_mae_green': 0.20,
    'relative_mae_yellow': 0.25,
    'r2_green': 0.80,
    'r2_yellow': 0.70,
    'tail_ratio_green': 2.0,   # p95 / MAE
    'tail_ratio_yellow': 2.5,
    'bias_abs_pct_green': 3.0, # |overprediction_rate - 50|
    'bias_abs_pct_yellow': 7.0,
}

summary = model_report['summary']
bias = model_report['bias']

tail_ratio = summary['error_p95'] / max(summary['test_mae'], 1e-9)
bias_abs_pct = abs(bias['overprediction_rate_percent'] - 50.0)

def traffic(value, green_ok, yellow_ok, higher_is_better=True):
    if higher_is_better:
        if value >= green_ok: return '??'
        if value >= yellow_ok: return '??'
        return '??'
    else:
        if value <= green_ok: return '??'
        if value <= yellow_ok: return '??'
        return '??'

checks = pd.DataFrame([
    {
        'metric': 'Relative MAE (MAE / median price)',
        'value': summary['relative_mae_vs_median_price'],
        'status': traffic(summary['relative_mae_vs_median_price'], THRESHOLDS['relative_mae_green'], THRESHOLDS['relative_mae_yellow'], higher_is_better=False),
        'target': f"<= {THRESHOLDS['relative_mae_green']:.2f} (yellow <= {THRESHOLDS['relative_mae_yellow']:.2f})"
    },
    {
        'metric': 'R2',
        'value': summary['test_r2'],
        'status': traffic(summary['test_r2'], THRESHOLDS['r2_green'], THRESHOLDS['r2_yellow'], higher_is_better=True),
        'target': f">= {THRESHOLDS['r2_green']:.2f} (yellow >= {THRESHOLDS['r2_yellow']:.2f})"
    },
    {
        'metric': 'Tail risk ratio (p95 / MAE)',
        'value': tail_ratio,
        'status': traffic(tail_ratio, THRESHOLDS['tail_ratio_green'], THRESHOLDS['tail_ratio_yellow'], higher_is_better=False),
        'target': f"<= {THRESHOLDS['tail_ratio_green']:.2f} (yellow <= {THRESHOLDS['tail_ratio_yellow']:.2f})"
    },
    {
        'metric': 'Bias imbalance |overprediction rate - 50|',
        'value': bias_abs_pct,
        'status': traffic(bias_abs_pct, THRESHOLDS['bias_abs_pct_green'], THRESHOLDS['bias_abs_pct_yellow'], higher_is_better=False),
        'target': f"<= {THRESHOLDS['bias_abs_pct_green']:.1f}pp (yellow <= {THRESHOLDS['bias_abs_pct_yellow']:.1f}pp)"
    },
])

# CV-test consistency from previous check (if available)
if 'best_cv_mae' in summary and 'best_cv_r2' in summary:
    mae_gap = summary.get('mae_gap_test_minus_cv', np.nan)
    r2_gap = summary.get('r2_gap_test_minus_cv', np.nan)
    checks = pd.concat([checks, pd.DataFrame([{
        'metric': 'CV/Test MAE gap (test - cv)',
        'value': mae_gap,
        'status': '??' if mae_gap <= 0 else '??',
        'target': '<= 0 preferred'
    }, {
        'metric': 'CV/Test R2 gap (test - cv)',
        'value': r2_gap,
        'status': '??' if r2_gap >= 0 else '??',
        'target': '>= 0 preferred'
    }])], ignore_index=True)

status_counts = checks['status'].value_counts()
if status_counts.get('??', 0) > 0:
    verdict = 'NO-GO (fix red items)'
elif status_counts.get('??', 0) > 1:
    verdict = 'CAUTION (review yellow items)'
else:
    verdict = 'GO (metrics look production-viable)'

print('=== TRAFFIC-LIGHT CHECK ===')
display(checks)
print('Verdict:', verdict)


=== TRAFFIC-LIGHT CHECK ===


,metric,value,status,target
0,Relative MAE (MAE / median price),0.093849,??,<= 0.20 (yellow <= 0.25)
1,R2,0.935162,??,>= 0.80 (yellow >= 0.70)
2,Tail risk ratio (p95 / MAE),4.280916,??,<= 2.00 (yellow <= 2.50)
3,Bias imbalance |overprediction rate - 50|,0.934277,??,<= 3.0pp (yellow <= 7.0pp)
4,CV/Test MAE gap (test - cv),-191.603300,??,<= 0 preferred
5,CV/Test R2 gap (test - cv),0.009352,??,>= 0 preferred


Verdict: NO-GO (fix red items)
